# Ingest drivers .json file
- 1.- Read the file using spark dataframe reader API 
- 2.- Define and enforce schem (preserve the nested structure)
- 3.- Add Metadata Columns
-     Source File
-     Ingestion Timestamp
- 4.- Write to bronze delta table 

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers  

In [0]:
# Define source_file and table_name
source_file = f"{landing_folder_path}/drivers.json"
table_name = f"{catalog_name}.{bronze_schema}.drivers"

### 1.- Read the file using spark dataframe reader API

In [0]:
# Define the Schema
from pyspark.sql.types import StructType, StructField, StringType, DateType

name_schema = StructType([
    StructField('givenName', StringType()),
    StructField('familyName', StringType())
])       

driver_schema = StructType([
    StructField('driverId', StringType()),
    StructField('name', name_schema),
    StructField('dateOfBirth', DateType()),
    StructField('nationality', StringType()),
    StructField('url', StringType()) 
])
                         

### 2.- Define and enforce schem (preserve the nested structure)


In [0]:
# Read data from the drivers file
drivers_df = (
    spark.read
    .format('json')
    .schema(driver_schema)
    .option('mode', 'FAILFAST')
    .load(source_file)
)

### 3.- Add Metadata Columns

In [0]:
drivers_final_df = add_ingestion_metadata(drivers_df)

### 4.- Write to bronze delta table

In [0]:
(
    drivers_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)

In [0]:
display(spark.read.table(table_name))